# Burgers PDE conditioning in function space and weight space

This notebook conditions a space-time GP on the viscous Burgers equation

$$u_t + u\,u_x = \nu\,u_{xx}, \qquad x \in [-1, 1],\ t \in [0, 1),\ \nu = 0.02,$$

with zero Dirichlet boundaries $u(\pm 1, t) = 0$, starting from a GP posterior given a few noisy observations of the sine initial condition $u(x, 0) = -\sin(\pi x)$. It does so in two ways: a **function-space** flow over the $N = H \times W$ grid values, with the PDE residual formed by finite differences, and a **weight-space** flow over the weights of $R$ random Fourier features, whose basis functions have analytic derivatives. The last section compares how the wall time of each scales with $N$.

## 1. Setup

The unit grid maps to space $x \in [-1, 1]$ along $H$ and time $t \in [0, 1)$ along $W$. Unlike `build_grid_in_box`, the local `build_grid` includes both ends of the space axis, so the Dirichlet conditions can be pinned at $x = \pm 1$ exactly; the spacing is $\Delta x = 2 / (H - 1)$. `R` is the number of random Fourier features used by the weight-space flow.

In [ ]:
import sys
import time
from pathlib import Path

import gpytorch
import matplotlib.pyplot as plt
import torch

from flow_inference.conditions import (
    EqualityCondition,
    PointWiseCondition,
    ProductCondition,
)
from flow_inference.flows import BasisGaussianProcessFlow, GaussianProcessFlow
from flow_inference.gp_utils import DecoupledRFFBasis, gp_posterior
from flow_inference.guidance import GuidanceMC, GuidancePiGDM
from flow_inference.sampler import IntegratorSampler

sys.path.insert(0, str(Path.cwd().parent))  # examples/, for plotting.py
from plotting import show_fields

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
H = 50               # space points
W = 20               # time points
dim = 2              # inputs (x, t)
num_data = 5         # initial-condition observations
noise = 0.01**2      # observation noise variance
R = 256              # random Fourier features
num_samples = 5
SEED = 0

torch.manual_seed(SEED)


def build_grid(H, W, device):
    """(H*W, 2) unit-box coordinates; space includes both ends, so x = 2*c - 1 spans [-1, 1]."""
    xs = torch.linspace(0.0, 1.0, H, device=device)
    ts = torch.arange(W, device=device) / W
    xx, tt = torch.meshgrid(xs, ts, indexing="ij")
    return torch.stack([xx, tt], dim=-1).view(-1, 2)

## 2. Data and GP posterior

### 2.1 Initial-condition data

`num_data` noisy observations of $u(x, 0) = -\sin(\pi x)$, all at $t = 0$.

In [ ]:
x_initial = torch.tensor([i / num_data for i in range(num_data)]).to(device)  # grid coordinate of x
Y_initial = -torch.sin(torch.pi * (2 * x_initial - 1.0))
Y_initial = Y_initial + noise**0.5 * torch.randn_like(Y_initial)  # noise is a variance
X_initial = torch.stack([x_initial, torch.zeros_like(x_initial)], dim=1)  # (num_data, 2), at t = 0

### 2.2 GP posterior

The GP has an anisotropic RBF kernel with lengthscales set by hand, and `gp_posterior` conditions it on the observations above. This posterior is the starting distribution of both flows below.

In [ ]:
coords = build_grid(H, W, device)  # (H*W, 2)
kernel = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel(ard_num_dims=2)).to(device)
kernel.base_kernel.lengthscale = torch.tensor([0.025, 0.3]).to(device)
kernel.outputscale = 1.0
p0_mean_fn, p0_kernel = gp_posterior(kernel, X_initial, Y_initial, noise=noise)

### 2.3 Prior samples

Samples from the function-space flow, given the initial condition only: no physics, no boundaries. Each panel is one field, with space vertically and time horizontally.

In [ ]:
flow = GaussianProcessFlow(p0_kernel, p0_mean_fn, coords, device=device)
sampler = IntegratorSampler(flow=flow)
uncond_samples = sampler.sample(num_samples, guidance=None, num_steps=1_000)
show_fields(uncond_samples.view(-1, H, W)[:5], cmap="bwr")

## 3. Burgers PDE and boundary conditions

`BurgersCondition` is an `EqualityCondition` on the PDE residual

$$r = u_t + u\,u_x - \nu\,u_{xx},$$

evaluated by centred differences at the interior grid points. A `PointWiseCondition` pins the first and last spatial rows ($x = \pm 1$) to zero at all times, using the mask from `dirichlet_mask`.

Both are imposed at every grid cell, so both carry quadrature `weights`: the cell area $\Delta x\,\Delta t$ for the PDE residual and the time step $\Delta t$ for the boundary rows. The log-likelihoods then approximate

$$-\frac{1}{2\sigma^2}\int r^2\,dx\,dt \qquad\text{and}\qquad -\frac{1}{2\sigma^2}\int u(\pm 1, t)^2\,dt,$$

so each $\sigma$ is a tolerance on a root-mean-square residual and means the same thing at any resolution. This matters for the cost comparison in Section 7, which refines $H$ thirty-two-fold: without the weights, the finer grids would also enforce the PDE more tightly.

In [ ]:
class BurgersCondition(EqualityCondition):
    """Burgers residual by finite differences, for a single-channel field in (H, W) order."""

    def __init__(self, H, W, sigma=10.0):
        dx = 2.0 / (H - 1)  # the grid includes x = -1 and x = 1
        dt = 1.0 / W
        super().__init__(sigma=sigma, weights=dx * dt)  # quadrature weight: one cell
        self.H = H
        self.W = W
        self.dx = dx
        self.dt = dt
        self.nu = 0.02  # viscosity

    def residual(self, x):  # x: (B, 1, H*W)
        B, C, N = x.shape
        assert C == 1, f"Expected C=1, got C={C}"
        assert N == self.H * self.W, f"Expected N={self.H*self.W}, got N={N}"

        u = x.view(B, self.H, self.W)
        u_int = u[:, 1:-1, 1:-1]  # interior points, (B, H-2, W-2)
        du_dt = (u[:, 1:-1, 2:] - u[:, 1:-1, :-2]) / (2.0 * self.dt)
        du_dx = (u[:, 2:, 1:-1] - u[:, :-2, 1:-1]) / (2.0 * self.dx)
        d2u_dx2 = (u[:, 2:, 1:-1] - 2.0 * u_int + u[:, :-2, 1:-1]) / (self.dx * self.dx)
        res = du_dt + u_int * du_dx - self.nu * d2u_dx2
        return res.reshape(B, 1, -1)  # (B, 1, (H-2)*(W-2))


def dirichlet_mask(H, W):
    """True on the first and last spatial rows (x = -1 and x = 1), for all times; (H*W,)."""
    mask = torch.zeros((H, W), dtype=torch.bool)
    mask[0, :] = True
    mask[-1, :] = True
    return mask.view(-1)

`ProductCondition` imposes both. On this $50 \times 20$ grid, $\sigma = 4.5\times10^{-7}$ for the PDE and $2.2\times10^{-7}$ for the boundaries correspond to per-point tolerances of $10^{-5}$ and $10^{-6}$.

In [ ]:
pde_condition = BurgersCondition(H, W, sigma=4.5e-7).to(device)
boundary_condition = PointWiseCondition(
    torch.zeros(H * W, device=device), dirichlet_mask(H, W), sigma=2.2e-7, weights=1.0 / W).to(device)
condition = ProductCondition([pde_condition, boundary_condition]).to(device)

## 4. Conditioned sampling in function space

`GuidanceMC` estimates the guidance term at each step from one Monte Carlo draw of the clean field given the current state.

In [ ]:
guidance = GuidanceMC(flow, condition, num_mc_samples=1).to(device)
samples = sampler.sample(num_samples, guidance=guidance, num_steps=1_000)
show_fields(samples.view(samples.shape[0], H, W)[:5], cmap="bwr")

## 5. Weight-space flow

The same problem, but the flow runs over basis weights rather than the $N$ grid values.

### 5.1 Conditioned RFF basis

A sampled function is $f(x) = w^\top\phi(x) + m(x)$ for a weight vector $w$, a feature vector $\phi(x)$ and a mean $m(x)$. `DecoupledRFFBasis` builds $\phi$ and $m$ by pathwise conditioning (Wilson et al., 2020): random Fourier features approximate the GP prior, and an exact update conditions it on the initial-condition data, so the initial condition enters through the basis rather than through a constraint. The latent width is $R_f + M$, where the extra $M$ features carry the $M$ initial-condition observations.

In [ ]:
basis = DecoupledRFFBasis(
    num_features=R, input_dim=dim, kernel=kernel,
    train_x=X_initial, train_y=Y_initial, noise=noise, device=device,
)
print(f"latent width R_f + M = {basis.num_features} + {basis.M} = {basis.R}")

### 5.2 Prior samples

Unguided draws from the weight-space flow, mapped back to the grid. They already respect the initial condition, but know nothing about the PDE or the boundaries.

In [ ]:
weight_flow = BasisGaussianProcessFlow(basis, coords, output_shape=(H, W), device=device)
weight_sampler = IntegratorSampler(flow=weight_flow)
uncond_weight_samples = weight_sampler.sample(num_samples, guidance=None, num_steps=1_000)
show_fields(uncond_weight_samples.view(-1, H, W)[:5], cmap="bwr")

### 5.3 Conditions in weight space

The same PDE and boundary conditions, written as functions of the weights $w$. `BurgersWeightCondition` uses the basis's analytic derivatives instead of finite differences: $u_x = w^\top\partial_x\phi + \partial_x m$, and likewise for $u_{xx}$ and $u_t$, where the factor `sx` converts derivatives in the unit-box coordinate to derivatives in $x$. The features and their derivatives are precomputed at the interior grid points, so evaluating the residual costs a few matrix products. Both conditions are evaluated on the same grid as in Section 3, so they carry the same quadrature weights and tolerances.

In [ ]:
class BurgersWeightCondition(EqualityCondition):
    """Burgers residual in weight space, using the basis's analytic derivatives."""

    def __init__(self, basis, grid, H, W, sigma=10.0):
        dx, dt = 2.0 / (H - 1), 1.0 / W
        super().__init__(sigma=sigma, weights=dx * dt)  # quadrature weight: one cell
        self.nu = 0.02  # viscosity
        sx = 0.5  # d/dx = sx * d/dcoord, since x = 2*coord - 1

        interior = torch.zeros((H, W), dtype=torch.bool)
        interior[1:-1, 1:-1] = True
        interior = interior.view(-1)  # (N,)

        self.phi = basis.phi(grid)[:, interior]  # (R, A)
        self.dphi_dx = sx * basis.compute_derivative(grid, (0,), (1,))[:, interior]
        self.dphi_dx2 = sx**2 * basis.compute_derivative(grid, (0,), (2,))[:, interior]
        self.dphi_dt = basis.compute_derivative(grid, (1,), (1,))[:, interior]

        self.mean = basis.mean(grid)[interior]  # (A,)
        self.dmean_dx = sx * basis.mean_derivative(grid, (0,), (1,))[interior]
        self.dmean_dx2 = sx**2 * basis.mean_derivative(grid, (0,), (2,))[interior]
        self.dmean_dt = basis.mean_derivative(grid, (1,), (1,))[interior]

    def residual(self, w):  # w: (B, 1, R)
        f = w @ self.phi + self.mean
        f_x = w @ self.dphi_dx + self.dmean_dx
        f_xx = w @ self.dphi_dx2 + self.dmean_dx2
        f_t = w @ self.dphi_dt + self.dmean_dt
        return f_t + f * f_x - self.nu * f_xx  # (B, 1, A)


class PointWiseWeightCondition(EqualityCondition):
    """Pin the field to a target at masked grid points, in weight space."""

    def __init__(self, target_field, mask, basis, grid, sigma=1e-6, weights=None):
        super().__init__(sigma=sigma, weights=weights)
        self.target_field = target_field
        self.register_buffer("index", mask.nonzero().squeeze(-1))  # integer indices, moved by .to()
        self.phi = basis.phi(grid)
        self.mean = basis.mean(grid)

    def residual(self, w):  # w: (B, 1, R)
        f = w @ self.phi + self.mean
        return f.index_select(-1, self.index) - self.target_field.index_select(-1, self.index)


weight_boundary_condition = PointWiseWeightCondition(
    torch.zeros(H * W, device=device), dirichlet_mask(H, W), basis, coords,
    sigma=2.2e-7, weights=1.0 / W).to(device)
weight_pde_condition = BurgersWeightCondition(basis, coords, H, W, sigma=4.5e-7).to(device)
weight_condition = ProductCondition([weight_boundary_condition, weight_pde_condition])

### 5.4 Conditioned sampling

Guided sampling of the weights, as in Section 4; the flow maps the sampled weights back to fields on the grid.

In [ ]:
weight_guidance = GuidanceMC(weight_flow, weight_condition, num_mc_samples=1).to(device)
weight_samples = weight_sampler.sample(num_samples, guidance=weight_guidance, num_steps=1_000)
show_fields(weight_samples, cmap="bwr")

## 6. Π-GDM guidance

`GuidanceMC` steers each sample by the gradient of the condition at a guess of the clean field, ignoring how uncertain that guess still is. With tolerances as tight as these ($\sigma \approx 10^{-7}$) that gradient is huge, so Sections 4 and 5.4 take 1,000 steps.

`GuidancePiGDM` accounts for that uncertainty: it implements Π-GDM (Song et al., 2023), the guidance of the spatial-prediction paper by Zammit-Mangion et al. (2026). At each step the clean latent given the current state $z_t$ is Gaussian, with mean $\mu_t = \alpha_t z_t$ and variance $v_t = 1 - \alpha_t^2$. Linearising the residual $r$ (here the PDE and boundary residuals together) around $\mu_t$ makes the guidance a Gaussian calculation,

$$\nabla \log p(y \mid z_t) \approx -\alpha_t\, J^\top \left(v_t\, J J^\top + \Sigma\right)^{-1} r(\mu_t),$$

where $J$ is the residual's Jacobian and $\Sigma$ the squared tolerances. While $v_t$ is large the pull is gentle, and it sharpens only as the sample settles, so the drift stays bounded however small $\sigma$ is: there is no clipping, and 25 steps are enough. Here each step linearises once (`num_iterations=1`, the default); more iterations move the linearisation point towards the most likely clean field, which lowers the residual further at proportionally higher cost. Each step solves a least-squares problem (by LSMR, using only Jacobian–vector products), so it costs more than a `GuidanceMC` step, but far fewer steps are needed.

Below, both guidances draw the same number of samples from the same seed, in function space and in weight space. Each run is timed and scored by the root-mean-square finite-difference PDE residual and the largest boundary value $|u(\pm 1, t)|$.

In [ ]:
def timed_sample(sampler, guidance, num_samples, **kwargs):
    """Guided samples, and the wall time taken to draw them."""
    torch.manual_seed(SEED)
    if device == "cuda":
        torch.cuda.synchronize()  # time the work, not the asynchronous launch
    start = time.perf_counter()
    out = sampler.sample(num_samples, guidance=guidance, verbose=False, **kwargs)
    if device == "cuda":
        torch.cuda.synchronize()
    return out, time.perf_counter() - start


def scores(fields):
    """RMS finite-difference PDE residual, and the largest |u| on the boundary rows."""
    fields = fields.reshape(fields.shape[0], 1, H * W)
    pde = pde_condition.residual(fields).pow(2).mean().sqrt().item()
    edge = fields[:, 0, dirichlet_mask(H, W).to(fields.device)].abs().max().item()
    return pde, edge


runs = {}
for space, (smp, fl, cond, mc) in {
    "function space": (sampler, flow, condition, guidance),
    "weight space": (weight_sampler, weight_flow, weight_condition, weight_guidance),
}.items():
    runs[space, "GuidanceMC, 1000 steps"] = timed_sample(smp, mc, num_samples, num_steps=1_000)
    pigdm = GuidancePiGDM(fl, cond).to(device)
    runs[space, "GuidancePiGDM, 25 steps"] = timed_sample(smp, pigdm, num_samples, num_steps=25)

for (space, label), (out, seconds) in runs.items():
    pde, edge = scores(out)
    print(f"{space:15s} {label:24s} {seconds:5.2f}s   RMS residual {pde:.3f}   max |u(±1)| {edge:.1e}")

The least-squares samples, in function space and then in weight space.

In [ ]:
show_fields(runs["function space", "GuidancePiGDM, 25 steps"][0].view(-1, H, W), cmap="bwr")
show_fields(runs["weight space", "GuidancePiGDM, 25 steps"][0].view(-1, H, W), cmap="bwr")

## 7. Cost

Both flows are run at increasing grid size $N = H \times W$, with $W$ fixed and $H$ growing, and `num_steps=250`.

* **Function space** pays an $\mathcal{O}(N^3)$ Cholesky factorisation once, stores the $(N, N)$ factor, and then pays an $\mathcal{O}(N^2)$ matrix product per step.
* **Weight space** keeps a state of width $R_f + M$, so its cost is linear in $N$.

`run_function_space` and `run_weight_space` build each flow and its conditions at a given $H$, time the setup and the sampling separately, and return the samples.

In [ ]:
BENCH_STEPS = 250    # per-step cost is what scales; the runs above use 1_000
BENCH_H = [50, 100, 200, 400, 800, 1600]
FUNC_BUDGET = 30.0   # seconds; skip function space once its next setup would exceed this


def _sync():
    if device == "cuda":
        torch.cuda.synchronize()  # time the work, not the asynchronous launch


def run_function_space(H, W):
    crd = build_grid(H, W, device)
    _sync()
    t0 = time.perf_counter()
    mu, kk = gp_posterior(kernel, X_initial, Y_initial, noise=noise)
    fl = GaussianProcessFlow(kk, mu, crd, device=device)  # O(N^3) Cholesky, O(N^2) storage
    _sync()
    t_setup = time.perf_counter() - t0

    cond = ProductCondition([
        BurgersCondition(H, W, sigma=4.5e-7),
        PointWiseCondition(torch.zeros(H * W, device=device), dirichlet_mask(H, W),
                           sigma=2.2e-7, weights=1.0 / W),
    ]).to(device)
    gd = GuidanceMC(fl, cond, num_mc_samples=1).to(device)

    _sync()
    t0 = time.perf_counter()
    out = IntegratorSampler(flow=fl).sample(num_samples, guidance=gd, num_steps=BENCH_STEPS, verbose=False)
    _sync()
    return t_setup, time.perf_counter() - t0, out.reshape(num_samples, 1, H * W)


def run_weight_space(H, W):
    crd = build_grid(H, W, device)
    _sync()
    t0 = time.perf_counter()
    bs = DecoupledRFFBasis(num_features=R, input_dim=dim, kernel=kernel,
                           train_x=X_initial, train_y=Y_initial, noise=noise, device=device)
    fl = BasisGaussianProcessFlow(bs, crd, output_shape=(H, W), device=device)
    _sync()
    t_setup = time.perf_counter() - t0

    cond = ProductCondition([
        BurgersWeightCondition(bs, crd, H, W, sigma=4.5e-7),
        PointWiseWeightCondition(torch.zeros(H * W, device=device), dirichlet_mask(H, W),
                                 bs, crd, sigma=2.2e-7, weights=1.0 / W),
    ]).to(device)
    gd = GuidanceMC(fl, cond, num_mc_samples=1).to(device)

    _sync()
    t0 = time.perf_counter()
    out = IntegratorSampler(flow=fl).sample(num_samples, guidance=gd, num_steps=BENCH_STEPS, verbose=False)
    _sync()
    return t_setup, time.perf_counter() - t0, out.reshape(num_samples, 1, H * W)

Run both at each grid size, scoring every run with the same finite-difference residual. Function space is skipped once it runs out of memory or its next setup would exceed `FUNC_BUDGET` seconds.

In [ ]:
results = {"func": [], "rff": []}
func_alive = True
for H_b in BENCH_H:
    N_b = H_b * W
    for name, runner in (("func", run_function_space), ("rff", run_weight_space)):
        if name == "func" and not func_alive:
            continue
        try:
            t_setup, t_sample, out = runner(H_b, W)
        except (RuntimeError, MemoryError) as err:  # the (N, N) factor no longer fits
            print(f"N = {N_b:6d}  {name:4s}  not run: {type(err).__name__}")
            func_alive = False
            continue
        # Score both with the same finite-difference residual.
        resid = BurgersCondition(H_b, W).to(device).residual(out).pow(2).mean().sqrt().item()
        results[name].append((N_b, t_setup, t_sample, resid))
        print(f"N = {N_b:6d}  {name:4s}  setup {t_setup:7.2f}s   "
              f"sample {t_sample:6.2f}s   residual {resid:.3e}")
        del out
        if device == "cuda":
            torch.cuda.empty_cache()
    # The Cholesky is O(N^3), so doubling H costs ~8x.
    if func_alive and results["func"] and results["func"][-1][1] * 8 > FUNC_BUDGET:
        print(f"N = {N_b:6d}  func  stopping: next setup would exceed {FUNC_BUDGET:.0f}s")
        func_alive = False

Plot total wall time against $N$, with dashed $\mathcal{O}(N^3)$ and $\mathcal{O}(N)$ reference lines through the last point of each method, and print the speed-up at the largest grid both methods reached.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for name, label, col, power in (("func", "function space", "tab:orange", 3),
                                ("rff", "RFF weight space", "tab:blue", 1)):
    Ns, setup, sample, _ = (list(v) for v in zip(*results[name], strict=False))
    total = [s + p for s, p in zip(setup, sample, strict=False)]
    ax.plot(Ns, total, marker="o", color=col, label=label)
    ax.plot(Ns, [total[-1] * (n / Ns[-1]) ** power for n in Ns], ls="--", lw=1.2,
            color=col, alpha=0.55, label=f"$O(N^{power})$, {label}" if power > 1 else f"$O(N)$, {label}")

ax.set(xscale="log", yscale="log", xlabel="N = H x W", ylabel="wall time (s)",
       title=f"Cost of one run ({BENCH_STEPS} steps, {num_samples} samples)")
ax.grid(True, which="both", ls="--", alpha=0.4)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

func_N, func_t = results["func"][-1][0], results["func"][-1][1] + results["func"][-1][2]
rff_t = next(s + p for n, s, p, _ in results["rff"] if n == func_N)
print(f"\nAt N = {func_N}, weight space is {func_t / rff_t:.0f}x faster for the same residual band.")
print(f"Function space stopped at N = {func_N}; weight space reached N = {results['rff'][-1][0]} "
      f"in {results['rff'][-1][1] + results['rff'][-1][2]:.1f}s.")

## 8. Summary

* The Burgers PDE and the Dirichlet boundaries are an `EqualityCondition` and a `PointWiseCondition`, combined with `ProductCondition`; weighting each by its cell size makes the tolerances $\sigma$ independent of the grid.
* Compare the conditioned samples in Sections 4 and 5.4 with the prior samples in Sections 2.3 and 5.2: guided samples should be pinned to zero along the top and bottom rows of each panel and evolve in time according to the PDE, rather than only following the GP's smoothness.
* The weight-space flow solves the same problem over $R_f + M$ basis weights, with the initial condition built into the basis and analytic derivatives in place of finite differences.
* `GuidancePiGDM` (Section 6) accounts for the uncertainty of the clean field, so its drift stays bounded under tight tolerances: 25 steps give a lower PDE residual than 1,000 steps of `GuidanceMC`, in slightly less time, in both spaces, and hold the boundaries closer to zero.
* In the cost plot, compare each curve with its dashed reference line: function-space cost grows much faster with $N$, and those runs stop once they exceed the time budget or memory, while weight space scales linearly.